# DriveSense-VLM — 03: Inference Benchmarks

Measures end-to-end inference performance for the production targets:
- **vLLM backend** — concurrent throughput on A100 (AWQ 4-bit model)
- **Local transformers backend** — sequential latency (T4-equivalent)
- **ViT-only benchmark** — TensorRT vs torch.compile vs eager

**GPU required**: A100 80GB (for vLLM; local backend works on T4)  
**Estimated time**: ~1 hour  
**Estimated cost**: ~12 CU  

**Prerequisites**: `02_optimization.ipynb` must have produced the quantized
model at `outputs/quantized_model/`.

**Production targets**:
| Metric | Target |
|--------|--------|
| E2E latency T4 (p50) | < 500 ms |
| E2E latency A100 (p50) | < 200 ms |
| ViT TensorRT p50 | < 25 ms |
| Throughput A100 | ≥ 8 fps |
| VRAM (T4) | < 6 GB |

## ⚠️ Before Running

Set **Runtime → Change runtime type → A100 GPU** for the vLLM benchmark.
The local backend section can run on T4 if you switch runtimes between sections.

In [ ]:
# Cell 3: Mount Drive + clone repo + symlinks + install
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/DriveSense-VLM'
os.makedirs(PROJECT_ROOT, exist_ok=True)

!git clone https://github.com/jayanth922/DriveSense-VLM.git /content/DriveSense-VLM 2>/dev/null || (cd /content/DriveSense-VLM && git pull --quiet)
%cd /content/DriveSense-VLM

!mkdir -p {PROJECT_ROOT}/data {PROJECT_ROOT}/outputs
!ln -sf {PROJECT_ROOT}/data /content/DriveSense-VLM/data 2>/dev/null || true
!ln -sf {PROJECT_ROOT}/outputs /content/DriveSense-VLM/outputs 2>/dev/null || true

print('Setup complete.')

In [ ]:
# Cell 4: Install vLLM and inference deps
# Upgrade setuptools first (Colab ships an outdated version that breaks PEP 660 editable installs)
!pip install --upgrade setuptools wheel build -q

# Project install — editable preferred, non-editable fallback
!pip install -e '.[inference]' -q 2>/dev/null || pip install '.[inference]' -q
!pip install vllm -q
print('vLLM and inference deps installed.')

In [ ]:
# Cell 5: Run vLLM benchmark (concurrent throughput on A100)
# Requires the quantized AWQ model at outputs/quantized_model/
from pathlib import Path

quant_dir = Path(PROJECT_ROOT) / 'outputs' / 'quantized_model'
if not quant_dir.exists():
    print('Quantized model not found — run 02_optimization.ipynb first.')
    print('Using --mock for demonstration ...')
    MOCK_FLAG = '--mock'
else:
    print(f'Quantized model found: {quant_dir}')
    MOCK_FLAG = ''

!python scripts/run_benchmark.py \
    --vllm \
    --model-dir {quant_dir} \
    --output {PROJECT_ROOT}/outputs/benchmarks/vllm_bench.json \
    {MOCK_FLAG}

In [ ]:
# Cell 6: Run ViT benchmark (TensorRT vs torch.compile vs eager)
!python scripts/run_benchmark.py \
    --vit-only \
    --output {PROJECT_ROOT}/outputs/benchmarks/vit_bench.json \
    {MOCK_FLAG}

In [ ]:
# Cell 7: Display results in a formatted table
import json
from pathlib import Path

bench_dir = Path(PROJECT_ROOT) / 'outputs' / 'benchmarks'

def _load(fname):
    p = bench_dir / fname
    return json.loads(p.read_text()) if p.exists() else {}

vllm = _load('vllm_bench.json')
vit  = _load('vit_bench.json')

print('=' * 60)
print('  Benchmark Results')
print('=' * 60)

def _row(label, value, target, unit=''):
    status = '✓' if value is not None and value != 'TBD' else '-'
    val_str = f'{value}{unit}' if value is not None else 'N/A'
    print(f'  {label:<35} {val_str:<12} target: {target}{unit}  {status}')

_row('E2E latency A100 p50', vllm.get('p50_ms'), 200, ' ms')
_row('E2E throughput A100', vllm.get('fps'), 8, ' fps')
_row('VRAM usage', vllm.get('vram_gb'), 6, ' GB')
_row('ViT p50 latency', vit.get('p50_ms'), 25, ' ms')

print('=' * 60)
print()
print('Full benchmark data:')
if vllm:
    print('  vLLM:', json.dumps(vllm, indent=4))

In [ ]:
# Cell 8: Run local (transformers) benchmark and save summary to Drive
# This simulates T4 deployment conditions
!python scripts/run_benchmark.py \
    --local \
    --output {PROJECT_ROOT}/outputs/benchmarks/local_bench.json \
    {MOCK_FLAG}

import json, datetime
from pathlib import Path

bench_dir = Path(PROJECT_ROOT) / 'outputs' / 'benchmarks'
summary = {
    'timestamp': datetime.datetime.utcnow().isoformat(),
    'vllm': json.loads((bench_dir / 'vllm_bench.json').read_text()) if (bench_dir / 'vllm_bench.json').exists() else {},
    'local': json.loads((bench_dir / 'local_bench.json').read_text()) if (bench_dir / 'local_bench.json').exists() else {},
    'vit': json.loads((bench_dir / 'vit_bench.json').read_text()) if (bench_dir / 'vit_bench.json').exists() else {},
}

summary_path = bench_dir / 'benchmark_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(f'Benchmark summary saved to {summary_path}')
print('\nProceed to 04_evaluation.ipynb')